In [ ]:
from time import time
import numpy as np
import os, glob
import torch_pca

import torch
import torch.nn.functional as F
import tifffile as tif
from concurrent.futures import ThreadPoolExecutor

device = "cuda:0"

path = "/nfs/scratch2/inacio/data/llsm/paper_data/to_run/"
n_components = 3

#####################

start = time()

def get_tif_shape(path):
    with tif.TiffFile(path) as tf:
        z = len(tf.pages)
        y, x = tf.pages[0].shape[:2]
    return (z,y,x)

def normalize(x):
    return (x-x.min())/(x.max()-x.min())

def to_8bit(x):
    return (normalize(x)*255).numpy().astype("uint8")

@torch.no_grad()
def get_pca(feats, n_components=1):
    og_shape = feats.shape[:-1]
    feats = feats.reshape(-1,feats.shape[-1])
    pca = torch_pca.PCA(n_components=n_components)

    fit = pca.fit(feats)
    output = pca.transform(feats).reshape(*og_shape,n_components)
    return output

@torch.no_grad()
def interpolate_pca(feats, target_shape):
    feats = feats.permute(3,0,1,2).unsqueeze(0)
    feats = F.interpolate(feats, size=target_shape, mode="trilinear", align_corners=False)
    feats = feats.squeeze(0).permute(1,2,3,0)
    return feats
    

folders = sorted(os.listdir(path))
for i, folder in enumerate(folders):
    if i % 1 == 0:
        print(i, len(folders), time()-start)
        
    load_path = os.path.join(path, folder)

    feats = torch.load(os.path.join(load_path, "lr_feats.pt"), weights_only=False).float().to(device)

    target_shape = get_tif_shape(os.path.join(load_path, "volume_unnorm.tif"))

    output = get_pca(feats, n_components=n_components)

    output = interpolate_pca(output, target_shape).cpu()

    output = np.stack([to_8bit(output[...,i]) for i in range(n_components)],axis=-1)
    
    save_path = os.path.join(load_path, f"PCA_{n_components}.tif")
    tif.imwrite(save_path, output)

print(time()-start)